# Kombinierte Labels: RBF-SVM

**Ziel:** Die RBF-SVM sagt Crop und Stage als gemeinsame Klasse `Crop|Stage` voraus. Ihre nichtlineare Entscheidungsgrenze wird gegen den Random Forest im selben Ansatz verglichen.

**Ablauf:** Die Hyperparameter werden ausschließlich per Cross-Validation auf dem Train-Anteil gewählt. Der Validierungssplit bleibt bis zum einmaligen Aufruf von `run()` unberührt.

In [ ]:
%load_ext autoreload
%autoreload 2

from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_combined_svm
from awp2.preprocessing import PreprocessingConfig

svm_preprocessing = PreprocessingConfig(scale=True)

## 1. Hyperparameter suchen

Die Suche vergleicht neun Kombinationen aus Regularisierung `C` und RBF-Bandbreite `gamma` mit fünf CV-Folds. Beide Werte werden ausschließlich anhand von `bacc_combined` auf dem Train-Anteil ausgewählt.

In [ ]:
tuned = tune(
    make_combined_svm(),
    TuneConfig(
        name="combined_svm_search",
        approach="combined",
        description="RBF-SVM auf kombinierten Crop/Stage-Labels: C und gamma vergleichen.",
        preprocessing=svm_preprocessing,
        param_grid={
            "estimator__C": [0.1, 1.0, 10.0],
            "estimator__gamma": ["scale", 0.001, 0.01],
        },
    ),
)

tuned.candidates.sort_values("rank")

## 2. Einmal validieren

Nur der nach CV ausgewählte Kandidat wird auf dem gemeinsamen Validierungssplit bewertet. Der Lauf verweist über `tuning_run` auf die Suche und wird lokal in MLflow gespeichert.

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="combined_svm",
        approach="combined",
        description="Beste RBF-SVM aus combined_svm_search, einmal auf der Validierung bewertet.",
        preprocessing=svm_preprocessing,
        tuning_run=tuned.run_id,
    ),
)

result.metrics.model_dump()

# 3. Analyse

In [ ]:
import pandas as pd
from IPython.display import display

from awp2.evaluation import plot_confusion_matrices
from awp2.plots import plot_class_recall, plot_top_confusions, plot_tuning_candidates

COMBINED_RF_BACC_COMBINED = 0.7787

comparison = pd.DataFrame(
    {
        "BAcc kombiniert": {
            "Random Forest": COMBINED_RF_BACC_COMBINED,
            "RBF-SVM": result.metrics.bacc_combined,
        },
        "Delta zum Random Forest": {
            "Random Forest": 0.0,
            "RBF-SVM": result.metrics.bacc_combined - COMBINED_RF_BACC_COMBINED,
        },
    }
)
display(comparison)

ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="estimator__C",
    validation_score=result.metrics.bacc_combined,
)
ax.set_title("Kombinierte Labels: CV-Score der RBF-SVM")
display(ax.figure)

for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

In [ ]:
{
    "tuning_run": tuned.run_id,
    "best_cv_score": tuned.best_score,
    "best_params": tuned.best_params,
    "evaluation_run": result.run_id,
}

In [ ]:
from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import save_doc_figure

tuning_ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="estimator__C",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title("Kombinierte Labels: CV-Score der RBF-SVM")
save_doc_figure(tuning_ax.figure, "combined_svm_tuning", MODEL_DOCS_FIGURES_DIR)

confusion_figure = plot_confusion_matrices(result.y_val, result.y_pred)
save_doc_figure(confusion_figure, "combined_svm_confusion", MODEL_DOCS_FIGURES_DIR)

recall_figure = plot_class_recall(result.y_val, result.y_pred)
save_doc_figure(recall_figure, "combined_svm_recall", MODEL_DOCS_FIGURES_DIR)

confusions_figure = plot_top_confusions(result.y_val, result.y_pred)
save_doc_figure(confusions_figure, "combined_svm_confusions", MODEL_DOCS_FIGURES_DIR)